In [2]:
import requests

customerid = "12345"
merchantid = "67890"
amount = 1000.0

url = "http://<YOUR_SERVER_IP>:8000/predict"
data = {
    'customer_id': customerid,
    'merchant_id': merchantid,
    'amount': amount
}

response = requests.post(url, json=data)
print(response.json())

{'fraud_probability': 0.0037532306741923094}


In [ ]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

# --- Connection details for the maintenance database ---
# NOTE: We connect to the default 'postgres' database, NOT the one we want to create.
db_params = {
    "host": "localhost",
    "user": "postgres",
    "password": "root" # Your superuser password
}

new_db_name = "testdb1"

try:
    # 1. Connect to the default 'postgres' database
    conn = psycopg2.connect(**db_params)
    
    # 2. Set the connection to AUTOCOMMIT mode
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    
    # 3. Create a cursor
    cur = conn.cursor()
    
    # 4. Execute the CREATE DATABASE command
    # Use %s placeholder for safety, even for the database name
    cur.execute(f"CREATE DATABASE {new_db_name};")
    
    print(f"✅ Database '{new_db_name}' created successfully.")

except psycopg2.errors.DuplicateDatabase:
    # This error (code '42P04') occurs if the database already exists.
    print(f"⚠️ Database '{new_db_name}' already exists.")

except (Exception, psycopg2.DatabaseError) as error:
    print(f"❌ Error creating database: {error}")

finally:
    # 5. Clean up by closing the cursor and connection
    if 'cur' in locals() and cur:
        cur.close()
    if 'conn' in locals() and conn:
        conn.close()

✅ Database 'logdb' created successfully.


In [7]:
import psycopg2

# Database connection parameters
# Replace with your actual database credentials
db_params = {
    "host": "localhost",
    "database": "testdb1",
    "user": "postgres",
    "password": "root" # The password you set during installation
}

try:
    # 1. Connect to the database
    conn = psycopg2.connect(**db_params)

    # 2. Create a cursor
    cur = conn.cursor()

    # 3. Define the CREATE TABLE query
    create_table_query = """
    CREATE TABLE IF NOT EXISTS users (
        id SERIAL PRIMARY KEY,
        username VARCHAR(50) NOT NULL UNIQUE,
        email VARCHAR(100) NOT NULL UNIQUE
    );
    """
    
    # ***** THE FIX IS HERE *****
    # You must execute the query to create the table
    cur.execute(create_table_query)
    print("✅ Table 'users' created or already exists.")


    # 4. Execute an INSERT statement
    username_to_add = 'charlie'
    email_to_add = 'charlie@example.com'
    
    # Using "ON CONFLICT" is a good practice to prevent errors if you run the script multiple times
    insert_query = """
    INSERT INTO users (username, email) VALUES (%s, %s)
    ON CONFLICT (username) DO NOTHING;
    """
    cur.execute(insert_query, (username_to_add, email_to_add))

    # Check if a row was actually inserted to give better feedback
    if cur.rowcount > 0:
        print(f"✅ User '{username_to_add}' inserted successfully.")
    else:
        print(f"ℹ️ User '{username_to_add}' already exists, no action taken.")


    # 5. Commit the transaction to make the changes permanent
    conn.commit()

    # 6. Execute a SELECT statement to fetch data
    cur.execute("SELECT id, username, email FROM users ORDER BY id;")

    # 7. Fetch all the results
    all_users = cur.fetchall()
    print("\n--- All Users in Database ---")
    for user in all_users:
        print(f"ID: {user[0]}, Username: {user[1]}, Email: {user[2]}")

    # 8. Close the cursor and connection
    cur.close()

except (Exception, psycopg2.DatabaseError) as error:
    print(f"❌ Error connecting to or working with PostgreSQL: {error}")

finally:
    if 'conn' in locals() and conn is not None:
        conn.close()
        print("\nDatabase connection closed.")


✅ Table 'users' created or already exists.
✅ User 'charlie' inserted successfully.

--- All Users in Database ---
ID: 1, Username: charlie, Email: charlie@example.com

Database connection closed.
